In [ ]:
# Peuple data/upsert/artists.csv depuis ChartMetric.
#
# Client "cache-first" : chaque endpoint n'est appelé que si l'artiste n'est pas
# déjà dans le CSV correspondant (data/raw/chartmetric/). Token :
# CHARTMETRIC_REFRESH_TOKEN dans notebooks/.env (ou .env à la racine).

import os
import sys

sys.path.append(os.path.abspath("../env"))

from chartmetric import ChartmetricClient, CsvStore, fetch_artist, rebuild_artists_csv, rebuild_genres_csv

client = ChartmetricClient()      # lit le refresh token, obtient un access token
store = CsvStore()                # data/raw/chartmetric/
print("Client ChartMetric prêt.")

In [ ]:
# Récupère (avec cache) toutes les données ChartMetric pour ces artistes.
# Écrit sous data/raw/chartmetric/ : artist_metadata, artist_stat, artist_geo,
# artist_audience_age / _gender / _country.

artists = [
    "Powerwolf", "Sabaton", "Babymetal", "Electric Callboy",
    "Ron Carter", "Wynton Marsalis", "Kamasi Washington",
    "Hans Zimmer", "Joe Hisaishi",
    # ajouter ici une superstar (ex. "Taylor Swift") et un petit artiste local
    # pour couvrir toute l'échelle de popularité
]

for name in artists:
    fetch_artist(client, name, store=store)          # refresh=True pour forcer le rappel API

In [ ]:
# Régénère data/upsert/artists.csv à partir du cache.
#  - audience / followers / popularité + cm_artist_score
#  - distribution d'âge = démographie d'audience réelle (Instagram/TikTok/YouTube),
#    sinon estimation par genre
# merge=True : les artistes absents du cache sont conservés.
# (genres.csv est maintenu à la main -> notebook 04 si besoin de le régénérer)

df_artists = rebuild_artists_csv(store)
df_artists

In [ ]:
# Inspection du cache brut (tout ce qui a été récupéré de ChartMetric)
import pandas as pd

for name in ["artists_index", "artist_metadata", "artist_stat", "artist_geo",
             "artist_audience_age", "artist_audience_gender", "artist_audience_country"]:
    df = store.load(name)
    print(f"{name:26s} {df.shape}")

store.load("artist_audience_age").head(12)